In [1]:
import pandas as pd
import numpy as np

current_sf_contacts = pd.read_csv("C:/Users/djlad/OneDrive/Desktop/HungerRush/sf_contacts.csv")
zoominfo_contacts = pd.read_csv("C:/Users/djlad/OneDrive/Desktop/HungerRush/zoominfo_contacts.csv")

In [2]:
#Keep only necessary columns from zoominfo_contacts
zoominfo_contacts = zoominfo_contacts[['First Name', 'Last Name', 'Email Address', 'Mobile phone', 'Company Name', 'Company Street Address', 'Company City', 'Company State', 'Company Zip Code', 
                                       'LinkedIn Contact Profile URL', 'Job Title']]

# Normalize emails (strip and lowercase) to avoid mismatches due to casing/spaces
zoominfo_contacts['Email Address'] = zoominfo_contacts['Email Address'].astype(str).str.strip().str.lower()
current_sf_contacts['Email'] = current_sf_contacts['Email'].astype(str).str.strip().str.lower()

#Drop known contacts
zoominfo_contacts = zoominfo_contacts[~zoominfo_contacts['Email Address'].isin(current_sf_contacts['Email'])]

# Keep job titles that contain 'chief', 'owner', 'founder', 'president' (case insensitive)
keywords = ['chief', 'owner', 'founder', 'president']
pattern = '|'.join(keywords)
zoominfo_contacts = zoominfo_contacts[zoominfo_contacts['Job Title'].str.contains(pattern, case=False, na=False)]

# Quick sanity check
print(f'Remaining zoominfo contacts: {len(zoominfo_contacts)}')

Remaining zoominfo contacts: 24


In [3]:
# add new columns to zoominfo_contacts to complete form upload format
zoominfo_contacts['Account ID'] = None
zoominfo_contacts['Campaign ID'] = None
zoominfo_contacts['Contact Owner ID'] = None
zoominfo_contacts['Lead Source'] = 'Marketing'

# import new csv that contains account IDs and Company Names.  Keeping only those two columns
account_ids = pd.read_csv("C:/Users/djlad/OneDrive/Desktop/HungerRush/sf_accounts.csv")
#Rename columns for easier reference
account_ids.rename(columns={'Account ID-18': 'Account ID ', 'Account Name': 'Company'}, inplace=True)
account_ids = account_ids[['Account ID ', 'Company']]

# Fuzzy-match `Company Name` to `Company` at >=80% similarity and bring in `Account ID` when matched.
from difflib import SequenceMatcher

def best_match(name, choices):
    name_s = str(name).strip().lower()
    best = (None, 0.0)
    for c in choices:
        c_s = str(c).strip().lower()
        if not c_s:
            continue
        ratio = SequenceMatcher(None, name_s, c_s).ratio()
        if ratio > best[1]:
            best = (c, ratio)
    return best  # (best_choice, best_ratio)

choices = account_ids['Company'].dropna().unique().tolist()
matches = {}
for comp in zoominfo_contacts['Company Name'].fillna('').unique():
    if not comp:
        matches[comp] = (None, 0.0)
        continue
    best_choice, best_ratio = best_match(comp, choices)
    matches[comp] = (best_choice if best_ratio >= 0.8 else None, best_ratio)

# Map matched company name into a helper column and merge to get Account ID
zoominfo_contacts['Company_matched'] = zoominfo_contacts['Company Name'].map(lambda x: matches.get(x, (None, 0.0))[0])
zoominfo_contacts = zoominfo_contacts.merge(account_ids[['Company','Account ID ']], how='left', left_on='Company_matched', right_on='Company')

# Report match statistics
matched_count = zoominfo_contacts['Company_matched'].notna().sum()
total = len(zoominfo_contacts)
print(f'Fuzzy-matched {matched_count} / {total} zoominfo rows (threshold >= 80%)')

Fuzzy-matched 30 / 34 zoominfo rows (threshold >= 80%)


In [4]:
# Convert 'Company State' to two-letter abbreviations
us_state_abbrev = {
    'Alabama': 'AL',
    'Alaska': 'AK',
    'Arizona': 'AZ',
    'Arkansas': 'AR',
    'California': 'CA',
    'Colorado': 'CO',
    'Connecticut': 'CT',
    'Delaware': 'DE',
    'Florida': 'FL',
    'Georgia': 'GA',
    'Hawaii': 'HI',
    'Idaho': 'ID',
    'Illinois': 'IL',
    'Indiana': 'IN',
    'Iowa': 'IA',
    'Kansas': 'KS',
    'Kentucky': 'KY',
    'Louisiana': 'LA',
    'Maine': 'ME',
    'Maryland': 'MD',
    'Massachusetts': 'MA',
    'Michigan': 'MI',
    'Minnesota': 'MN',
    'Mississippi': 'MS',
    'Missouri': 'MO',
    'Montana': 'MT',
    'Nebraska': 'NE',
    'Nevada': 'NV',
    'New Hampshire': 'NH',
    'New Jersey': 'NJ',
    'New Mexico': 'NM',
    'New York': 'NY',
    'North Carolina': 'NC',
    'North Dakota': 'ND',
    'Ohio': 'OH',
    'Oklahoma': 'OK',
    'Oregon': 'OR',
    'Pennsylvania': 'PA',
    'Rhode Island': 'RI',
    'South Carolina': 'SC',
    'South Dakota': 'SD',
    'Tennessee': 'TN',
    'Texas': 'TX',
    'Utah': 'UT',
    'Vermont': 'VT',
    'Virginia': 'VA',
    'Washington': 'WA',
    'West Virginia': 'WV',
    'Wisconsin': 'WI',
    'Wyoming': 'WY',
    'District Of Columbia': 'DC'
}

def convert_state(state):
    state = str(state).strip()
    if len(state) == 2 and state.isalpha():
        return state.upper()
    return us_state_abbrev.get(state.title(), state)
zoominfo_contacts['Company State'] = zoominfo_contacts['Company State'].apply(convert_state)

In [5]:
#Add contact owner ID to row based on first letter of company name
def classify(value):
    if not value or not isinstance(value, str):
        return None  

    first_char = value[0].upper()
    if ord(first_char) <= 77:  # 77 = 'M'
        return "005UU0000040N7JYAU"
    else:
        return "005UU000004YB5BYAW"
zoominfo_contacts['Contact Owner ID'] = zoominfo_contacts['Company Name'].apply(classify)

#Some zip codes begin with leading zeros that may have been dropped; ensure they are 5-digit strings
def format_zip(zip_code):
    zip_str = str(zip_code).strip()
    if zip_str.isdigit():
        return zip_str.zfill(5)
    return zip_str
zoominfo_contacts['Company Zip Code'] = zoominfo_contacts['Company Zip Code'].apply(format_zip)

In [6]:
#keep columns in the correct order for upload
zoominfo_contacts = zoominfo_contacts[['First Name', 'Last Name', 'Email Address', 'Mobile phone', 'Company Name', 'Campaign ID', 
                                       'Contact Owner ID', 'Company Street Address', 'Company City', 'Company State', 'Company Zip Code',
                                       'Lead Source', 'Account ID ', 'LinkedIn Contact Profile URL']]

#remove duplicates based on LinkedIn Contact Profile URL (assuming this is a unique identifier for contacts)
zoominfo_contacts.drop_duplicates(subset=['LinkedIn Contact Profile URL'], inplace=True)

#export to csv
zoominfo_contacts.to_csv("C:/Users/djlad/OneDrive/Desktop/HungerRush/zoominfo_contacts_to_upload_Feb10.csv", index=False)